<a href="https://colab.research.google.com/github/rtajeong/M4_2026/blob/main/lab_62_naver_movie_Tfidf_RNN_rev4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Naver Movie sentiment analysis
- using Tfidf vectorizer

- 감성분석
- 네이버 영화평점 (Naver sentiment movie corpus v.1.0) 데이터(https://github.com/e9t/nsmc)
- 영화 리뷰 20만건이 저장됨. 각 평가 데이터는 0(부정), 1(긍정)으로 label 됨.

한글 자연어 처리
- KoNLPy(“코엔엘파이”라고 읽습니다)는 한국어 정보처리를 위한 파이썬 패키지입니다.
- konlpy 패키지에서 제공하는 Twitter라는 문서 분석 라이브러리 사용 (트위터 분석 뿐 아니라 한글 텍스트
  처리도 가능)
- colab 사용 권장

In [1]:
!pip install konlpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.4/19.4 MB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 438.5/438.5 kB 14.7 MB/s eta 0:00:00


In [2]:
# 패키지 설치
import konlpy
import pandas as pd
import numpy as np
from konlpy.tag import Twitter, Okt
from sklearn.feature_extraction.text import TfidfVectorizer
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Activation, Input
from sklearn import model_selection, metrics

# 토큰 파서
def okt_tokenizer(text):
    return Okt().morphs(text)

In [3]:
okt_tokenizer("Welcome to data science world!...한국말도 똑 같아요...")

['Welcome',
 'to',
 'data',
 'science',
 'world',
 '!...',
 '한국말',
 '도',
 '똑',
 '같아요',
 '...']

```
다음 사이트는 더 이상 존재하지 않음. 대신에 직접 가져와야 함.

#!curl -L https://bit.ly/2X9Owwr -o ratings_train.txt
#!curl -L https://bit.ly/2WuLd5I -o ratings_test.txt
```

In [4]:
import pandas as pd

# 올바른 경우 (raw 파일 다운로드)
!wget https://raw.githubusercontent.com/e9t/nsmc/master/ratings_train.txt
!wget https://raw.githubusercontent.com/e9t/nsmc/master/ratings_test.txt

# 탭(\t) 구분자로 읽기
df_train = pd.read_csv("ratings_train.txt", sep="\t")
df_test = pd.read_csv("ratings_test.txt", sep="\t")


--2026-07-06 09:05:33--  https://raw.githubusercontent.com/e9t/nsmc/master/ratings_train.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 14628807 (14M) [text/plain]
Saving to: ‘ratings_train.txt’

ratings_train.txt   100%[===================>]  13.95M  --.-KB/s    in 0.09s   

2026-07-06 09:05:33 (158 MB/s) - ‘ratings_train.txt’ saved [14628807/14628807]

--2026-07-06 09:05:34--  https://raw.githubusercontent.com/e9t/nsmc/master/ratings_test.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.111.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 4893335 (4.7M) [application/octet-

In [5]:
df_train[:5]

,id,document,label
0,9976970,아 더빙.. 진짜 짜증나네요 목소리,0
1,3819312,흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나,1
2,10265843,너무재밓었다그래서보는것을추천한다,0
3,9045019,교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정,0
4,6483659,사이몬페그의 익살스런 연기가 돋보였던 영화!스파이더맨에서 늙어보이기만 했던 커스틴 ...,1


In [6]:
df_test[:5]

,id,document,label
0,6270596,굳 ㅋ,1
1,9274899,GDNTOPCLASSINTHECLUB,0
2,8544678,뭐야 이 평점들은.... 나쁘진 않지만 10점 짜리는 더더욱 아니잖아,0
3,6825595,지루하지는 않은데 완전 막장임... 돈주고 보기에는....,0
4,6723715,3D만 아니었어도 별 다섯 개 줬을텐데.. 왜 3D로 나와서 제 심기를 불편하게 하죠??,0


In [ ]:
# df_train['document'].values == df_train['document'].to_numpy() # all True

array([ True,  True,  True, ...,  True,  True,  True])

In [7]:
text_train, y_train = df_train['document'].to_numpy(), df_train['label'].to_numpy()
text_test, y_test = df_test['document'].to_numpy(), df_test['label'].to_numpy()

In [8]:
text_train.shape, y_train.shape, text_test.shape, y_test.shape

((150000,), (150000,), (50000,), (50000,))

In [9]:
# too much... -> let's take few of them
text_train, y_train = text_train[:2000], y_train[:2000]
text_test, y_test = text_test[:1000], y_test[:1000]

In [10]:
y_train.shape, y_test.shape

((2000,), (1000,))

In [11]:
y_train.mean(), y_test.mean()    # check distribution of classes 1 and 0

(np.float64(0.4945), np.float64(0.508))

In [12]:
cv = TfidfVectorizer(tokenizer=okt_tokenizer, max_features=3000)
X_train = cv.fit_transform(text_train)
X_test = cv.transform(text_test)

/usr/local/lib/python3.12/dist-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [13]:
X_train.shape, y_train.shape, X_test.shape, y_test.shape

((2000, 3000), (2000,), (1000, 3000), (1000,))

In [15]:
cv.get_feature_names_out()[:5], cv.get_feature_names_out()[-5:]

(array(['!', '!!', '!!!', '!!!!', '!!!!!!!!!!!!!!'], dtype=object),
 array(['힘들건데', '힘들게', '힘들다', '힘들었네요', '힘들었음'], dtype=object))

# Linear Classification

In [16]:
from sklearn.linear_model import SGDClassifier
clf = SGDClassifier()
result = clf.fit(X_train, y_train)
feature_names = cv.get_feature_names_out()
print("Training : ", result.score(X_train, y_train))
print("Testing : ", result.score(X_test, y_test))

Training :  0.984
Testing :  0.746


# 로지스틱 회귀

In [17]:
from sklearn.linear_model import LogisticRegression
lr = LogisticRegression()
result = lr.fit(X_train,y_train)
feature_names = cv.get_feature_names_out()
print("Training : ", result.score(X_train, y_train))
print("Testing : ", result.score(X_test, y_test))

Training :  0.92
Testing :  0.771


# MLP

In [21]:
from tensorflow.keras.utils import to_categorical
to_categorical(y_train)

array([[1., 0.],
       [0., 1.],
       [1., 0.],
       ...,
       [1., 0.],
       [1., 0.],
       [1., 0.]])

In [22]:
# use one-hot encoded target
from tensorflow.keras.utils import to_categorical

y_train_ohe = to_categorical(y_train)
y_test_ohe = to_categorical(y_test)

max_words = X_train.shape[1]   # 3000

model = Sequential()
model.add(Input(shape=(max_words,)))
model.add(Dense(64, activation="relu"))
model.add(Dropout(0.2))
model.add(Dense(64, activation="relu"))
model.add(Dropout(0.2))
model.add(Dense(2, activation='softmax'))

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

model.fit(X_train.toarray(), y_train_ohe, epochs=5, batch_size=100)
print("Training : ", model.evaluate(X_train.toarray(), y_train_ohe))
print("Testing : ", model.evaluate(X_test.toarray(), y_test_ohe))

Epoch 1/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.5350 - loss: 0.6921
Epoch 2/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7870 - loss: 0.6705
Epoch 3/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8705 - loss: 0.5854
Epoch 4/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8915 - loss: 0.4012
Epoch 5/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9405 - loss: 0.2278
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.9635 - loss: 0.1466
Training :  [0.14657093584537506, 0.9635000228881836]
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.7700 - loss: 0.4977
Testing :  [0.49772050976753235, 0.7699999809265137]


In [23]:
# use binary target

max_words = X_train.shape[1]

model = Sequential()
model.add(Input(shape=(max_words,)))
model.add(Dense(64, activation="relu"))
model.add(Dropout(0.2))
model.add(Dense(64, activation="relu"))
model.add(Dropout(0.2))
model.add(Dense(1, activation='sigmoid'))

model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

model.fit(X_train.toarray(), y_train, epochs=5, batch_size=100)

print("Training : ", model.evaluate(X_train.toarray(), y_train))
print("Testing : ", model.evaluate(X_test.toarray(), y_test))

Epoch 1/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.5365 - loss: 0.6920
Epoch 2/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.7700 - loss: 0.6773
Epoch 3/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8455 - loss: 0.6213
Epoch 4/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.8895 - loss: 0.4797
Epoch 5/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9165 - loss: 0.3154
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9465 - loss: 0.2118
Training :  [0.2118040770292282, 0.9465000033378601]
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7670 - loss: 0.4811
Testing :  [0.4811345636844635, 0.7670000195503235]


# RNN

In [24]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((2000, 3000), (1000, 3000), (2000,), (1000,))

In [25]:
# RNN 학습을 위한 데이터 재배열
X_train_rnn = X_train.toarray().reshape((X_train.shape[0], 1, X_train.shape[1]))
X_test_rnn = X_test.toarray().reshape((X_test.shape[0], 1, X_test.shape[1]))

print(X_train_rnn.shape, X_test_rnn.shape)

(2000, 1, 3000) (1000, 1, 3000)


In [26]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense

model = Sequential()
model.add(Input(shape=(X_train_rnn.shape[1], X_train_rnn.shape[2])))
model.add(SimpleRNN(128, return_sequences=True))  # timesteps=1 이므로 특별한 의미는 없음
model.add(Activation('relu'))
model.add(Dropout(0.2))
model.add(SimpleRNN(128))
model.add(Activation('relu'))
model.add(Dropout(0.2))
model.add(Dense(2, activation="softmax"))
model.summary()

model.compile(loss='categorical_crossentropy',
              optimizer='adam', metrics=['accuracy'])

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn (SimpleRNN)          │ (None, 1, 128)         │       400,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 1, 128)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 1, 128)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_1 (SimpleRNN)        │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 2)              │           258 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 433,666 (1.65 MB)

 Trainable params: 433,666 (1.65 MB)

 Non-trainable params: 0 (0.00 B)

In [27]:
model.fit(X_train_rnn, y_train_ohe, batch_size = 100, epochs=5)

Epoch 1/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step - accuracy: 0.5400 - loss: 0.6889
Epoch 2/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.8230 - loss: 0.6268
Epoch 3/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8820 - loss: 0.4272
Epoch 4/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9315 - loss: 0.2173
Epoch 5/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9720 - loss: 0.1061


In [28]:
y_pred = np.argmax(model.predict(X_test_rnn), axis=1)
print("accuracy score: ", metrics.accuracy_score(y_test, y_pred))
print("Training : ", model.evaluate(X_train_rnn, y_train_ohe))
print("Testing : ", model.evaluate(X_test_rnn, y_test_ohe))

32/32 ━━━━━━━━━━━━━━━━━━━━ 2s 26ms/step
accuracy score:  0.763
63/63 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - accuracy: 0.9875 - loss: 0.0625
Training :  [0.06251509487628937, 0.987500011920929]
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.7630 - loss: 0.5941
Testing :  [0.5941399335861206, 0.7630000114440918]


# LSTM
- https://colah.github.io/posts/2015-08-Understanding-LSTMs/

In [30]:
from keras.layers import LSTM
model = Sequential()
model.add(Input(shape=(X_train_rnn.shape[1], X_train_rnn.shape[2])))
model.add(LSTM(128, return_sequences=True))
model.add(Activation('relu'))
model.add(Dropout(0.2))
model.add(LSTM(128))
model.add(Activation('relu'))
model.add(Dropout(0.2))
model.add(Dense(2, activation='softmax'))

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

In [31]:
model.fit(X_train_rnn, y_train_ohe, batch_size = 100, epochs=5)

Epoch 1/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step - accuracy: 0.5100 - loss: 0.6930
Epoch 2/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.5330 - loss: 0.6901
Epoch 3/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.8330 - loss: 0.6684
Epoch 4/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.8820 - loss: 0.5799
Epoch 5/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.9045 - loss: 0.3954


In [32]:
y_pred = np.argmax(model.predict(X_test_rnn), axis=1)
print("accuracy score: ", metrics.accuracy_score(y_test, y_pred))
print("Training : ", model.evaluate(X_train_rnn, y_train_ohe))
print("Testing : ", model.evaluate(X_test_rnn, y_test_ohe))

32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step
accuracy score:  0.765
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9325 - loss: 0.2684
Training :  [0.268380731344223, 0.9325000047683716]
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7650 - loss: 0.4893
Testing :  [0.4892774522304535, 0.7649999856948853]


- time_step = 1 : 매번 하나의 토큰(혹은 벡터)만 들어오기 때문에 순환구조를 사용할 기회가 없다. 따라서, 겉보기에는 MLP(일반 feed-forward 네트워크)와 유사하다.
- 하지만, RNN은 구조적으로 hidden state를 유지할 수 있는 recurrent connection을 갖고 있다.따라서 여러 step을 이어서 학습할 때는 여전히 과거 hidden state를 전달할 수 있다. (하지만, 새로운 Batch 가 시작될 때 hidden state는 모두 0(또는 초기값)으로 다시 초기화된다)
  - 딥러닝에서는 batch를 단순히 병렬 계산을 위한 묶음으로 취급한다.
  - Keras 의 경우 stateful=False(기본값) → batch 사이의 정보는 전달되지 않는다.
  - stateful=True 를 사용하면 batch 간 상태를 이어갈 수 있지만, 데이터를 순서대로 배치하고 상태를 적절히 초기화(reset_states())하는 등 특별한 관리가 필요하다.
- tfidf 는 단어의 등장 빈도와 중요도를 반영하지만, 단어의 순서(시퀀스 정보)는 완전히 사라진다. 즉, RNN이 기대하는 시간적/순차적 구조가 없다.

# Exercise

### 한국어 불용어 처리
- 한국어  불용어 확인은 형태소 분석 라이브러리인 KonLPy 를 이용하면 됨.
- (예) 한국어 품사 중 조사를 추출하는 예
- pos (part-of-speech): 품사 (명사, 동사, ...)

In [33]:
from konlpy.tag import Twitter

In [34]:
Twitter().morphs("텍스트 데이터를 이용해서 불용어 사전을 구축하기 위한 간단 예제")

/usr/local/lib/python3.12/dist-packages/konlpy/tag/_okt.py:17: UserWarning: "Twitter" has changed to "Okt" since KoNLPy v0.4.5.
  warn('"Twitter" has changed to "Okt" since KoNLPy v0.4.5.')


['텍스트',
 '데이터',
 '를',
 '이용',
 '해서',
 '불',
 '용어',
 '사전',
 '을',
 '구축',
 '하기',
 '위',
 '한',
 '간단',
 '예제']

In [35]:
Twitter().pos("텍스트 데이터를 이용해서 불용어 사잔을 구축하기 위한 간단 예제")

[('텍스트', 'Noun'),
 ('데이터', 'Noun'),
 ('를', 'Josa'),
 ('이용', 'Noun'),
 ('해서', 'Verb'),
 ('불', 'Noun'),
 ('용어', 'Noun'),
 ('사잔', 'Noun'),
 ('을', 'Josa'),
 ('구축', 'Noun'),
 ('하기', 'Verb'),
 ('위', 'Noun'),
 ('한', 'Josa'),
 ('간단', 'Noun'),
 ('예제', 'Noun')]

In [36]:
Twitter().pos("텍스트 데이터를 이용해서 불용어 사전을 구축하기 위한 간단 예제", norm=True)   # norm - 오타 수정 (사잔->사전)

[('텍스트', 'Noun'),
 ('데이터', 'Noun'),
 ('를', 'Josa'),
 ('이용', 'Noun'),
 ('해서', 'Verb'),
 ('불', 'Noun'),
 ('용어', 'Noun'),
 ('사전', 'Noun'),
 ('을', 'Josa'),
 ('구축', 'Noun'),
 ('하기', 'Verb'),
 ('위', 'Noun'),
 ('한', 'Josa'),
 ('간단', 'Noun'),
 ('예제', 'Noun')]

In [37]:
Twitter().nouns("텍스트 데이터를 이용해서 불용어 사전을 구축하기 위한 간단 예제")  # 명사 추출

['텍스트', '데이터', '이용', '불', '용어', '사전', '구축', '위', '간단', '예제']

- norm: 오타수정, stem: 어근 찾기

In [38]:
from konlpy.tag import Twitter

word_tags = Twitter().pos("텍스트 데이터를 이용해서 불용어 사전을 구축하기 위한 간단 예제", norm=True, stem=True)
print(word_tags)
stop_words = [word[0] for word in word_tags if word[1]=="Josa"]
print (stop_words)

[('텍스트', 'Noun'), ('데이터', 'Noun'), ('를', 'Josa'), ('이용', 'Noun'), ('하다', 'Verb'), ('불', 'Noun'), ('용어', 'Noun'), ('사전', 'Noun'), ('을', 'Josa'), ('구축', 'Noun'), ('하다', 'Verb'), ('위', 'Noun'), ('한', 'Josa'), ('간단', 'Noun'), ('예제', 'Noun')]
['를', '을', '한']


### Pickling

- Python 객체를 통째로 저장하고 복원하는 바이너리 직렬화 방식이다. 머신러닝 모델, 전처리기, NumPy 배열 등 Python 객체를 그대로 저장하는 데 적합하다.
- 저장되는 것은:
  - 모델 구조
  - 학습된 weight
  - 하이퍼파라미터
  - 내부 변수
  - Python 객체 상태, 등 거의 모든 것.
- in sklearn,
  - pickle.dump(model,f), pickle.load(f) 또는
  - joblib.dump(model,"rf.pkl"), joblib.load("rf.pkl") (preferred in sklearn)
- in keras,
  - model.save("model.keras"), keras.models.load_model()
- in pytorch,
  - torch.save(model,"model.pt"), torch.load()

In [41]:
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
cv_ = TfidfVectorizer(tokenizer=okt_tokenizer, max_features=10)

In [42]:
import os
import pickle
if not os.path.isfile("X_data.pickle"):
    print('file does not exist')
    X_data_pre = cv_.fit_transform(text_train)
    pickle.dump(X_data_pre, open("X_data.pickle", "wb"))

file does not exist


/usr/local/lib/python3.12/dist-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [43]:
# 저장된 tfidf vector 데이터 읽기
with open('X_data.pickle', 'rb') as f:
    X_data_post = pickle.load(f)

In [44]:
X_data_pre.toarray() == X_data_post.toarray()

array([[ True,  True,  True, ...,  True,  True,  True],
       [ True,  True,  True, ...,  True,  True,  True],
       [ True,  True,  True, ...,  True,  True,  True],
       ...,
       [ True,  True,  True, ...,  True,  True,  True],
       [ True,  True,  True, ...,  True,  True,  True],
       [ True,  True,  True, ...,  True,  True,  True]])

- BoW (Bag of Words)
  - document-term matrix
  - tfidf matrix

In [46]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
corpus = [
        'This is the first document.',
        'This document is the second document.',
        'And this is the third one.',
        'Is this the first document?',
]
vectorizer1 = CountVectorizer()
X = vectorizer1.fit_transform(corpus)
print(vectorizer1.get_feature_names_out())
print(X.toarray())

['and' 'document' 'first' 'is' 'one' 'second' 'the' 'third' 'this']
[[0 1 1 1 0 0 1 0 1]
 [0 2 0 1 0 1 1 0 1]
 [1 0 0 1 1 0 1 1 1]
 [0 1 1 1 0 0 1 0 1]]


In [47]:
vectorizer2 = TfidfVectorizer()
X = vectorizer2.fit_transform(corpus)
print(vectorizer2.get_feature_names_out())
print(X.toarray().round(2))

['and' 'document' 'first' 'is' 'one' 'second' 'the' 'third' 'this']
[[0.   0.47 0.58 0.38 0.   0.   0.38 0.   0.38]
 [0.   0.69 0.   0.28 0.   0.54 0.28 0.   0.28]
 [0.51 0.   0.   0.27 0.51 0.   0.27 0.51 0.27]
 [0.   0.47 0.58 0.38 0.   0.   0.38 0.   0.38]]


----------------------------------